In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim

In [3]:
spark = get_sparkSession("Load silver.fact_payments")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "fact_payments"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_fact_payments.ipynb"
_bronze_table = "bronze.fact_payments"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.fact_payments


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze.raw_customer and de-duplicating the data

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

df_dedup = df.withColumn("_rn", expr("row_number() over (partition by payment_id, order_id sort by created_on desc)")) \
             .withColumnRenamed("status", "payment_status") \
             .where("_rn = 1") \
             .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Bronze Table Data count : 2
SPARK-APP: Table count after de-dupe : 2


In [10]:
df_dedup.show(10)
df_dedup.printSchema()


+----------+--------+------------+-----------+------+-------+--------------+--------+------+--------------+---------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|payment_id|order_id|payment_date|customer_id| store|channel|payment_method|currency|amount|payment_status|is_refund|refund_amount|          created_on|          created_by|         modified_on|         modified_by|      source_file|
+----------+--------+------------+-----------+------+-------+--------------+--------+------+--------------+---------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|   PAY1001|   O1001|    20250101|    CUST001|AMZ_US|    MKT|   Credit Card|     USD|   900|          Paid|    False|            0|2026-01-29 00:35:...|raw_fact_payments...|2026-01-29 00:35:...|raw_fact_payments...|fact_payments.csv|
|   PAY1002|   O1002|    20250102|    CUST002|  EBAY|    MKT|   

In [11]:
#Formating the data

df_ld = df_dedup.withColumn("payment_date", to_date("payment_date", 'yyyyMMdd')) \
                .withColumn("amount", expr("CAST(amount as decimal(18,2))")) \
                .withColumn("is_refund", when(lower("is_refund") == "true", True).otherwise(False)) \
                .withColumn("refund_amount", expr("CAST(refund_amount as decimal(18,2))"))                

In [12]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("payment_id", upper(trim("payment_id"))) \
             .withColumn("order_id", upper(trim("order_id"))) \
             .withColumn("customer_id", upper(trim("customer_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel"))) \
             .withColumn("payment_method", upper(trim("payment_method"))) \
             .withColumn("currency", upper(trim("currency"))) \
             .withColumn("payment_status", upper(trim("payment_status"))) 

In [13]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [14]:
df_ld.show(10)
df_ld.printSchema()

+----------+--------+------------+-----------+------+-------+--------------+--------+-------+--------------+---------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|payment_id|order_id|payment_date|customer_id| store|channel|payment_method|currency| amount|payment_status|is_refund|refund_amount|          created_on|          created_by|         modified_on|         modified_by|      source_file|
+----------+--------+------------+-----------+------+-------+--------------+--------+-------+--------------+---------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|   PAY1001|   O1001|  2025-01-01|    CUST001|AMZ_US|    MKT|   CREDIT CARD|     USD| 900.00|          PAID|    false|         0.00|2026-01-29 01:09:...|silver_fact_payme...|2026-01-29 01:09:...|silver_fact_payme...|fact_payments.csv|
|   PAY1002|   O1002|  2025-01-02|    CUST002|  EBAY|    MKT

In [15]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.fact_payments and load_controller


In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+----------+--------+------------+-----------+------+-------+--------------+--------+-------+--------------+---------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|payment_id|order_id|payment_date|customer_id| store|channel|payment_method|currency| amount|payment_status|is_refund|refund_amount|          created_on|          created_by|         modified_on|         modified_by|      source_file|
+----------+--------+------------+-----------+------+-------+--------------+--------+-------+--------------+---------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|   PAY1001|   O1001|  2025-01-01|    CUST001|AMZ_US|    MKT|   CREDIT CARD|     USD| 900.00|          PAID|    false|         0.00|2026-01-29 01:09:...|silver_fact_payme...|2026-01-29 01:09:...|silver_fact_payme...|fact_payments.csv|
|   PAY1002|   O1002|  2025-01-02|    CUST002|  EBAY|    MKT

In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+-------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|   table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+-------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|fact_payments|full-load| overwrite|2026-01-29 01:09:...|    success|           2|2026-01-29 01:09:...|silver_fact_payme...|2026-01-29 01:09:...|silver_fact_payme...|
+------+-----------+-------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |2            |
+-------+-------------+



In [19]:
spark.stop()